# Regularization Demo: Ridge (L2) & Lasso (L1)

This notebook demonstrates how regularization combats overfitting
by penalizing large coefficients.

**Mathematical formulation:**
- Ridge (L2): $\min_w \|y - Xw\|^2 + \alpha \|w\|_2^2$
- Lasso (L1): $\min_w \|y - Xw\|^2 + \alpha \|w\|_1$

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
from src.data_generation import generate_polynomial_data, true_function
from src.regularization import fit_ridge, fit_lasso, evaluate_regularization
from src.visualization import (
    plot_regularization_effect,
    plot_ridge_vs_lasso_coefficients,
    save_figure,
)

## 1. Setup: Overfit Scenario

We use degree=10 polynomial features on 30 data points — a recipe for overfitting.
Regularization will control the model complexity.

In [ ]:
X_train, y_train = generate_polynomial_data(n_samples=30, noise_std=0.3, seed=42)
X_test, y_test = generate_polynomial_data(n_samples=100, noise_std=0.0, seed=0)

# Reshape for sklearn
X_train_2d = X_train.reshape(-1, 1)
X_test_2d = X_test.reshape(-1, 1)

## 2. Ridge Regression: Effect of α

As α increases, coefficients shrink toward zero, reducing overfitting.
But too much regularization leads to underfitting.

In [ ]:
alphas = [1e-8, 1e-6, 1e-4, 1e-2, 1e-1, 1.0, 10.0, 100.0]
ridge_results = evaluate_regularization(
    X_train_2d, y_train, X_test_2d, y_test,
    degree=10, alphas=alphas, method="ridge",
)

fig = plot_regularization_effect(
    ridge_results["alphas"],
    ridge_results["train_errors"],
    ridge_results["test_errors"],
    ridge_results["coefficient_norms"],
    method="Ridge",
)
save_figure(fig, "../report/figures/ridge_regularization.png")
fig

## 3. Lasso Regression: Sparsity Effect

Lasso drives some coefficients exactly to zero, performing feature selection.

In [ ]:
lasso_results = evaluate_regularization(
    X_train_2d, y_train, X_test_2d, y_test,
    degree=10, alphas=alphas, method="lasso",
)

fig = plot_regularization_effect(
    lasso_results["alphas"],
    lasso_results["train_errors"],
    lasso_results["test_errors"],
    lasso_results["coefficient_norms"],
    method="Lasso",
)
save_figure(fig, "../report/figures/lasso_regularization.png")
fig

## 4. Ridge vs. Lasso: Coefficient Comparison

At a moderate α, Ridge shrinks all coefficients evenly, while Lasso zeros out irrelevant ones.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=10, include_bias=False)
X_train_poly = poly.fit_transform(X_train_2d)

ridge_fit = fit_ridge(X_train_poly, y_train, alpha=0.01)
lasso_fit = fit_lasso(X_train_poly, y_train, alpha=0.01)

feature_names = [f"x^{i+1}" for i in range(10)]
fig = plot_ridge_vs_lasso_coefficients(
    ridge_fit["coefficients"],
    lasso_fit["coefficients"],
    feature_names=feature_names,
)
save_figure(fig, "../report/figures/ridge_vs_lasso_coefficients.png")
fig